In [1]:
from AppKit import NSWorkspace, NSRunningApplication
from ApplicationServices import (
    AXUIElementCreateApplication,
    AXUIElementCreateSystemWide,
    AXUIElementCopyAttributeValue,
    AXUIElementCopyAttributeNames,
    AXUIElementCopyActionNames,
    AXUIElementSetAttributeValue,
    AXUIElementPerformAction,
    kAXErrorSuccess,
)

In [2]:
def ax_attr(element, attr):
    """Read a single AX attribute from an element"""
    err, value = AXUIElementCopyAttributeValue(element, attr, None)
    if err != kAXErrorSuccess:
        return None
    return value

def ax_all_attrs(element):
    """List all available attribute names on an element"""
    err, names = AXUIElementCopyAttributeNames(element, None)
    return names if err == kAXErrorSuccess else []

def ax_actions(element):
    """List all available actions on an element (e.g. AXPress, AXFocus)."""
    err, actions = AXUIElementCopyActionNames(element, None)
    return actions if err == kAXErrorSuccess else []

def ax_do(element, action):
    """Perform an AX action on an element."""
    return AXUIElementPerformAction(element, action)

In [3]:
def list_running_apps():
    """Print all running apps and their PIDs."""
    for app in NSWorkspace.sharedWorkspace().runningApplications():
        print(app.localizedName(), app.processIdentifier())
        
def get_ax_app_by_name(name: str):
    """
    Get an AX application element by display name.
    Fine for most apps, but bundle ID is safer for Figma (see below).
    """
    for app in NSWorkspace.sharedWorkspace().runningApplications():
        if app.localizedName() == name:
            return AXUIElementCreateApplication(app.processIdentifier())
    raise RuntimeError(f"App '{name}' not found or not running")

def get_ax_app_figma():
    """
    Get Figma's AX element via bundle ID — more reliable than matching by name
    since display names can vary between Figma versions.
    """
    
    FIGMA_BUNDLE_ID = "com.figma.Desktop"
    
    apps = NSRunningApplication.runningApplicationsWithBundleIdentifier_(FIGMA_BUNDLE_ID)
    if not apps:
        raise RuntimeError("Figma is not running")
    return AXUIElementCreateApplication(apps[0].processIdentifier())

def get_focused_element():
    """Get whatever UI element currently has keyboard focus, system-wide."""
    sys = AXUIElementCreateSystemWide()
    err, el = AXUIElementCopyAttributeValue(sys, "AXFocusedUIElement", None)
    return el if err == kAXErrorSuccess else None

In [4]:
def walk(element, depth=0, max_depth=6):
    """
    Print the AX tree from a given element downward.
    Use on a window element to see full UI hierarchy.
    Reduce max_depth if output is too large (Figma trees are deep).
    """
    if depth > max_depth:
        return

    role  = ax_attr(element, "AXRole")
    title = ax_attr(element, "AXTitle")
    desc  = ax_attr(element, "AXDescription")

    print("  " * depth, role, title or desc or "")

    for child in ax_attr(element, "AXChildren") or []:
        walk(child, depth + 1, max_depth)

In [5]:
def find_elements(element, role=None, title=None, found=None):
    """
    Search the AX tree for elements matching role and/or title.
    Either argument can be None to match anything.

    Example:
        find_elements(window, role="AXButton", title="Export")
        find_elements(window, role="AXStaticText")
    """
    if found is None:
        found = []

    r = ax_attr(element, "AXRole")
    t = ax_attr(element, "AXTitle")

    role_match  = (role  is None or r == role)
    title_match = (title is None or t == title)

    if role_match and title_match:
        found.append(element)

    for child in ax_attr(element, "AXChildren") or []:
        find_elements(child, role, title, found)

    return found

In [6]:
def inspect_element(element):
    """
    Print all attributes and actions available on a single element.
    Useful when you find an interesting element and want to know what it exposes.
    """
    print("--- Attributes ---")
    for attr in ax_all_attrs(element):
        value = ax_attr(element, attr)
        print(f"  {attr}: {value}")

    print("--- Actions ---")
    for action in ax_actions(element):
        print(f"  {action}")

In [ ]:
list_running_apps()

In [8]:
ax_app = get_ax_app_figma()

windows = ax_attr(ax_app, "AXWindows") or []
for i, w in enumerate(windows):
    print(i, ax_attr(w, "AXTitle"), ax_attr(w, "AXRole"))

0 Untitled AXWindow


In [9]:
if windows:
    main_window = next(
        (w for w in windows if ax_attr(w, "AXRole") == "AXWindow"),
        windows[0]
    )

    walk(main_window, max_depth=8)

 AXWindow Untitled
   AXGroup Untitled
     AXGroup 
       AXGroup 
         AXGroup 
           AXGroup 
             AXGroup 
               AXGroup 
             AXGroup 
               AXGroup 
                 AXWebArea Figma
               AXGroup 
                 AXWebArea Untitled – Figma


In [10]:
export_buttons = find_elements(main_window, role="AXButton", title="Export")
print(f"Found: {len(export_buttons)}")

Found: 1


In [11]:
text_els = find_elements(main_window, role="AXStaticText")
for el in text_els[:20]:
    print(" ", ax_attr(el, "AXValue"))

  Screenreader support for the board is currently disabled. To enable it via Accessibility settings, press ⌘K, type Accessibility Settings, and press Enter. To see available keyboard shortcuts, enter ⌃⇧Question mark .
  File name
  Drafts
  Rectangle 1
  100%
  Page
  Color
  %
  Toggle visibility
  Toolbelt Mode
